# 模型指标与典型错误样本分析

本 Notebook 汇总 ConvNeXt-Tiny 垃圾分类模型的测试指标、训练过程、错分方向和典型错误样本。

## 结论摘要

## 1. 数据与模型输出

- 最终使用模型数量：`1` 个，即 `convnext_tiny`。训练脚本支持 `convnext_tiny`、`efficientnet_b0`、`efficientnet_b3`、`resnet50` 四种架构选项，但本次最终产物和测试指标均来自 `convnext_tiny`。
- 模型做法：使用 ImageNet 预训练的 ConvNeXt-Tiny 做迁移学习，将最后分类头替换为 10 类输出；输入尺寸 `384 x 384`，训练 `20` 轮。
- 训练配置：CrossEntropyLoss（含 label smoothing）、AdamW 优化器、CosineAnnealingLR 学习率调度；训练集使用随机裁剪、翻转、旋转、颜色扰动和 Random Erasing 增强，验证/测试集只做 Resize 与 ImageNet Normalize。
- 测试集样本数：`917`，错分样本数：`84`。
- 测试集 Accuracy：`0.9084`，Macro-F1：`0.9176`，Weighted-F1：`0.9085`。
- Macro ROC-AUC OvR：`0.9834`，Weighted ROC-AUC OvR：`0.9822`。
- 本地训练日志中最佳验证准确率出现在第 `18` 轮，val_acc=`0.9169`；训练最终 train_acc 达到 `0.9990`。


In [ ]:
from pathlib import Path
import json
import pandas as pd

artifact_dir = Path('/Users/sylviachan/Desktop/機器學習/分組/model/artifacts/artifacts')
metrics = json.loads((artifact_dir / 'test_metrics.json').read_text(encoding='utf-8'))
errors = pd.read_csv(artifact_dir / 'misclassified_samples.csv')
train_log = pd.read_csv(artifact_dir / 'train_log.csv')
metrics['test_accuracy'], metrics['macro_f1'], len(errors)

## 模型各项指标测试与评估

本节汇总模型在独立测试集上的整体测试结果。测试集共 917 张图片，不参与训练和调参，因此可以用于评估模型泛化能力。

| 指标 | 数值 | 评估含义 |
|---|---:|---|
| Accuracy | 0.9084 | 整体正确率，约 90.84% 测试图片被正确分类。 |
| Macro Precision | 0.9209 | 每类 precision 等权平均，反映类别均衡视角下的误报控制。 |
| Macro Recall | 0.9157 | 每类 recall 等权平均，反映类别均衡视角下的漏报控制。 |
| Macro F1 | 0.9176 | 每类 F1 等权平均，说明模型对小类也保持较好表现。 |
| Weighted F1 | 0.9085 | 按样本数加权后的 F1，反映测试集真实分布下的综合性能。 |
| ROC-AUC Macro OvR | 0.9834 | 一对多 AUC 的简单平均，衡量概率排序和区分能力。 |
| ROC-AUC Weighted OvR | 0.9822 | 按类别样本数加权的一对多 AUC。 |

**总体评估：** Accuracy、Macro-F1、Weighted-F1 均在 0.91 左右，说明模型整体分类效果较好；AUC 接近 0.98，说明概率排序能力较强。但 `trash`、`plastic`、`cardboard` 等类别仍存在明显混淆，因此需要结合分类报告和错分样本继续分析。

In [ ]:
pd.read_csv('tables/overall_metrics.csv')

## 每类 ROC-AUC

![Per-class ROC-AUC](figures/per_class_auc.png)

**结果解读：** 多数类别 ROC-AUC 高于 0.97，说明模型总体区分能力强。`trash` 的 AUC 最低，约 0.9343；`cardboard` 约 0.9623，`plastic` 约 0.9744，说明这些类别不仅 top-1 分类容易错，概率排序也相对更困难。

**评估结论：** AUC 高说明模型能较好地区分目标类别与非目标类别，但不能替代 F1 和混淆矩阵。实际部署时仍应对 `trash/plastic/cardboard/paper` 等高风险类别做二次确认或 top-k 输出。

In [ ]:
pd.read_csv('tables/per_class_auc.csv')

## 图 1：分类指标

![Per-class metrics](figures/per_class_metrics.png)

**结果解读：** `plastic` 的 F1 最低，为 `0.8288`，`trash` 为 `0.8400`，`cardboard` 和 `paper` 也明显低于 `clothes`、`shoes`、`biological`。这说明模型不是整体失效，而是主要卡在材质相近、类别边界模糊的样本上。`plastic` recall 较低，表示真实塑料被漏判较多；`trash` 类由于定义宽泛，类内视觉差异最大。

**可能原因：** 衣服、鞋子、生物垃圾通常有较稳定的形状和纹理；纸、纸板、塑料、其他垃圾则可能在颜色、形状、背景上高度重叠。

**解决方案：** 针对低 F1 类别补充边界样本，并复核标签规则；训练时考虑 class-balanced sampling、focal loss 或提高困难类别采样权重。

In [ ]:
pd.read_csv('tables/per_class_metrics.csv').sort_values('f1')

## 图 2：训练曲线

![Training curves](figures/training_curves.png)

**结果解读：** 训练准确率从约 `0.7530` 提升到 `0.9990`，验证准确率最高达到 `0.9169`。第 10 轮以后验证集提升变慢并出现波动，说明继续训练的收益有限。训练集接近满分而验证集没有同步接近满分，说明后期存在轻微过拟合。

**可能原因：** 模型已经充分学习训练集，但训练集中某些纹理和背景特征未必能泛化到验证/测试样本。

**解决方案：** 使用最佳验证集 checkpoint；加入 early stopping、增强数据多样性、适当提高正则化或降低后期学习率。更关键的是补充困难样本，而不是单纯增加 epoch。

## 图 3：各真实类别错误率

![Error rate by class](figures/error_rate_by_class.png)

**结果解读：** `trash` 错误率最高，约 `19.23%`；`plastic` 约 `18.58%`；`cardboard` 约 `15.04%`。相比之下，`clothes`、`battery`、`shoes` 的错误率很低，说明错误集中在少数类别，而不是所有类别平均变差。

**可能原因：** `trash` 是兜底类别，视觉形态不统一；`plastic` 有瓶、袋、杯、膜等多种形态；`cardboard` 和 `paper` 在单张图片中很难通过颜色和纹理区分。

**解决方案：** 重点复核 `trash` 标签定义；对 `plastic/cardboard/trash` 做难例挖掘，把错分样本加入下一轮训练或建立重点验证子集。

## 图 4：主要错分方向

![Top error pairs](figures/top_error_pairs.png)

**结果解读：** 最大错分方向是 `cardboard -> paper`，共 `11` 次；`paper -> cardboard` 共 `6` 次，说明纸与纸板存在双向混淆。`trash -> plastic`、`trash -> paper`、`plastic -> glass`、`plastic -> trash` 各有 `5` 次，也说明塑料、玻璃、其他垃圾之间边界不稳定。

**可能原因：** 模型可能过度依赖颜色、反光和平面纹理，而没有充分捕捉厚度、瓦楞结构、透明材质差异等细粒度信息。

**解决方案：** 对高频混淆对补充区分性样本；部署时对这些类别对的低置信预测返回 top-2 或触发二次确认。

In [ ]:
pd.read_csv('tables/top_error_pairs.csv')

## 图 5：重构混淆矩阵

![Confusion matrix](figures/reconstructed_confusion_matrix.png)

**结果解读：** 混淆矩阵对角线整体明显，说明模型总体分类能力较好。非对角线热点集中在 `paper/cardboard/plastic/trash` 附近，与前面错误率和错分方向的结论一致。

**可能原因：** ConvNeXt-Tiny 的整体特征提取能力足够，瓶颈主要来自少数类别边界和数据质量，而不是模型完全学不会。

**解决方案：** 保持当前 ConvNeXt-Tiny 主干，把优化资源集中到非对角线热点类别；下一轮评估继续比较这些热点格子是否下降。

## 图 6：高置信错分样本

![High-confidence errors](figures/high_confidence_error_samples.png)

**结果解读：** 高置信错分共有 `38` 个，占全部错分样本 `45.2%`。这类错误比低置信错误更危险，因为模型不仅错，而且很确定；单纯设置置信度阈值无法解决这部分问题。

**可能原因：** 标签边界模糊、物体只露出局部、背景干扰、反光、遮挡或材质高度相似，都可能让模型形成错误但稳定的判断依据。

**解决方案：** 逐张人工复核高置信错分样本，标注错误原因；标签错则修正，样本合理但困难则加入 hard example set 重训。部署时对高频混淆对增加二次校验。

In [ ]:
pd.read_csv('tables/high_confidence_error_samples.csv')[['filename', 'true_label', 'pred_label', 'confidence', 'source']]

## 模型数量与实现方式

- 最终使用 `1` 个模型：`ConvNeXt-Tiny`。
- 训练脚本支持 `convnext_tiny`、`efficientnet_b0`、`efficientnet_b3`、`resnet50` 四种架构选项，但本次最终评估结果来自 `convnext_tiny`。
- 做法是迁移学习：加载 ImageNet 预训练 ConvNeXt-Tiny，将最后分类头替换成 10 类输出。
- 训练使用 PyTorch/Torchvision，损失函数为 CrossEntropyLoss，优化器为 AdamW，学习率调度为 CosineAnnealingLR，训练 20 轮并保存验证集最优模型。
- 评估阶段在独立测试集上计算 Accuracy、Precision、Recall、F1、ROC-AUC、混淆矩阵和错分样本。

## 基于分析的模型优化与协作影响控制

基于当前错误分析，可以继续优化模型，但不建议直接覆盖现有 `best_model.pt`。这个文件可能已经被后端推理或前端展示流程引用，直接替换会影响其他同学。

**安全优化方案：** 保留当前模型作为 v1，新实验放到独立目录，例如 `model/artifacts/convnext_tiny_v2/`，新权重命名为 `best_model_v2.pt`，并生成对应的 `test_metrics_v2.json`、`classification_report_v2.txt` 和 `misclassified_samples_v2.csv`。

**优化重点：** 优先处理 `paper/cardboard/plastic/trash`。数据上补充边界样本和高置信错分样本；训练上尝试 class-balanced sampling、focal loss 和 early stopping；部署上对低置信样本返回 top-k，对高频混淆对做二次确认。

**切换条件：** 只有当 v2 在困难类别上明显提升，且整体 Accuracy/Macro-F1 不下降，并确认 `class_names.json` 顺序、输入尺寸和推理接口不变后，再通知后端/前端同学切换模型。

详细方案见 `analysis/optimization_plan.md`。

## 改进方向

- 增加 paper/cardboard/plastic/trash 的困难样本。
- 对高置信错分样本进行人工复核，排查标签噪声。
- 引入 class-balanced sampling、focal loss 或类别定向增强。
- 部署阶段使用置信度阈值或 top-k 结果，降低低置信样本的误判风险。